In [3]:
# TODO: is the folder nuscenes-multimodal-learning/data necessary? 
# TODO: clean folder structure
# TODO: clean gitignore

# Objective

Train a first LiDAR-only baseline on the full nuScenes dataset using MMDetection3D.

This baseline will serve as the reference model before trying any fusion approach.
The goal of this notebook is to document the setup, data preparation, config choice,
training launch, monitoring, evaluation, and first interpretation of results.

Scope for this first baseline:
- use a standard LiDAR-only MMDetection3D config
- keep modifications minimal
- use the full nuScenes dataset
- save logs and checkpoints in the project space

# Imports

In [4]:
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import sys
import platform

# Paths and constants

In [5]:
# ===== Main paths =====
REPO_ROOT: Path = Path("/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning")
MMDET3D_ROOT: Path = REPO_ROOT / "external" / "mmdetection3d"
DATA_ROOT: Path = Path("/rs_scratch/users/ae04q066/nuscenes_full")

# ===== Project subfolders =====
EXTERNAL_DIR: Path = REPO_ROOT / "external"
DATA_DIR: Path = REPO_ROOT / "data"
DATA_LINK: Path = DATA_DIR / "nuscenes"
NOTEBOOKS_DIR: Path = REPO_ROOT / "notebooks"
CONFIGS_DIR: Path = REPO_ROOT / "configs"
WORK_DIR_ROOT: Path = REPO_ROOT / "work_dirs"
SCRIPTS_DIR: Path = REPO_ROOT / "scripts"

# ===== MMDetection3D data path =====
MMDET3D_DATA_DIR: Path = MMDET3D_ROOT / "data"
MMDET3D_DATA_LINK: Path = MMDET3D_DATA_DIR / "nuscenes"

# ===== Experiment naming =====
EXPERIMENT_NAME: str = "lidar_baseline_nuscenes"
EXPERIMENT_WORK_DIR: Path = WORK_DIR_ROOT / EXPERIMENT_NAME

# ===== Shared constants =====
EXPECTED_RAW_DIRS: List[str] = [
    "maps",
    "samples",
    "sweeps",
    "v1.0-trainval",
]

EXPECTED_PROCESSED_ITEMS: List[str] = [
    "nuscenes_infos_train.pkl",
    "nuscenes_infos_val.pkl",
    "nuscenes_dbinfos_train.pkl",
    "nuscenes_gt_database",
]

timestamp: str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
python_version: str = sys.version.split("\n")[0]
platform_name: str = platform.platform()

print("Timestamp:", timestamp)
print()
print("Python version:", python_version)
print("Platform:", platform_name)
print()
print("REPO_ROOT:", REPO_ROOT)
print("MMDET3D_ROOT:", MMDET3D_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("DATA_LINK:", DATA_LINK)
print("MMDET3D_DATA_LINK:", MMDET3D_DATA_LINK)
print("WORK_DIR_ROOT:", WORK_DIR_ROOT)
print("EXPERIMENT_WORK_DIR:", EXPERIMENT_WORK_DIR)

Timestamp: 2026-04-07 21:38:56

Python version: 3.8.20 (default, Oct  3 2024, 15:24:27) 
Platform: Linux-5.14.0-611.36.1.el9_7.x86_64-x86_64-with-glibc2.17

REPO_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
DATA_ROOT: /rs_scratch/users/ae04q066/nuscenes_full
DATA_LINK: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes
MMDET3D_DATA_LINK: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes
WORK_DIR_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/work_dirs
EXPERIMENT_WORK_DIR: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/work_dirs/lidar_baseline_nuscenes


# Path checks

In [6]:
paths_to_check: Dict[str, Path] = {
    "REPO_ROOT": REPO_ROOT,
    "MMDET3D_ROOT": MMDET3D_ROOT,
    "DATA_ROOT": DATA_ROOT,
    "EXTERNAL_DIR": EXTERNAL_DIR,
    "DATA_DIR": DATA_DIR,
    "DATA_LINK": DATA_LINK,
    "MMDET3D_DATA_DIR": MMDET3D_DATA_DIR,
    "MMDET3D_DATA_LINK": MMDET3D_DATA_LINK,
    "NOTEBOOKS_DIR": NOTEBOOKS_DIR,
    "CONFIGS_DIR": CONFIGS_DIR,
    "SCRIPTS_DIR": SCRIPTS_DIR,
    "WORK_DIR_ROOT": WORK_DIR_ROOT,
}

for name, path in paths_to_check.items():
    print(f"{name:18s} exists={path.exists()}  path={path}")

REPO_ROOT          exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT       exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
DATA_ROOT          exists=True  path=/rs_scratch/users/ae04q066/nuscenes_full
EXTERNAL_DIR       exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external
DATA_DIR           exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data
DATA_LINK          exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes
MMDET3D_DATA_DIR   exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data
MMDET3D_DATA_LINK  exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes
NOTEBOOKS_DIR      exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/notebo

# Environment notes

- Raw nuScenes data is stored on scratch.
- Code, notebooks, logs, and checkpoints are stored in the project space.
- This notebook documents and checks the pipeline.
- Actual preprocessing, training, and testing are performed with MMDetection3D scripts.

# Dataset layout check

Before running MMDetection3D preprocessing, verify that the full nuScenes dataset
exists at the expected root and contains the standard raw folders.

This helps catch path mistakes early and confirms that the dataset is ready for
the MMDetection3D pipeline.

In [7]:
print("DATA_ROOT:", DATA_ROOT)
print()

for folder_name in EXPECTED_RAW_DIRS:
    folder_path: Path = DATA_ROOT / folder_name
    print(f"{folder_name:15s} exists={folder_path.exists()}  path={folder_path}")

DATA_ROOT: /rs_scratch/users/ae04q066/nuscenes_full

maps            exists=True  path=/rs_scratch/users/ae04q066/nuscenes_full/maps
samples         exists=True  path=/rs_scratch/users/ae04q066/nuscenes_full/samples
sweeps          exists=True  path=/rs_scratch/users/ae04q066/nuscenes_full/sweeps
v1.0-trainval   exists=True  path=/rs_scratch/users/ae04q066/nuscenes_full/v1.0-trainval


# Project-side dataset symlink

To keep the raw nuScenes data on scratch while making it easy to access from the project,
I create a project-level symbolic link:

`data/nuscenes -> /rs_scratch/users/ae04q066/nuscenes_full`

```/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes``` -> ```/rs_scratch/users/ae04q066/nuscenes_full```

In [8]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_LINK.exists() or DATA_LINK.is_symlink():
    print("Project data link already exists:", DATA_LINK)
    print("Resolved target:", DATA_LINK.resolve())
else:
    DATA_LINK.symlink_to(DATA_ROOT, target_is_directory=True)
    print("Created project data symlink:")
    print(DATA_LINK)
    print("->", DATA_LINK.resolve())

Project data link already exists: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes
Resolved target: /rs_scratch/users/ae04q066/nuscenes_full


In [9]:
print("DATA_LINK exists:", DATA_LINK.exists())
print("DATA_LINK is symlink:", DATA_LINK.is_symlink())
print("DATA_LINK resolves to:", DATA_LINK.resolve())
print()

for folder_name in EXPECTED_RAW_DIRS:
    folder_path: Path = DATA_LINK / folder_name
    print(f"{folder_name:15s} exists={folder_path.exists()}  path={folder_path}")

DATA_LINK exists: True
DATA_LINK is symlink: True
DATA_LINK resolves to: /rs_scratch/users/ae04q066/nuscenes_full

maps            exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/maps
samples         exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/samples
sweeps          exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/sweeps
v1.0-trainval   exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/data/nuscenes/v1.0-trainval


# MMDetection3D-side dataset symlink

The standard MMDetection3D command uses `./data/nuscenes` relative to the
MMDetection3D repository root.

To stay close to that workflow, I also create a symbolic link inside
`external/mmdetection3d/data/` that points to the project dataset link.

In [10]:
MMDET3D_DATA_DIR.mkdir(parents=True, exist_ok=True)

if MMDET3D_DATA_LINK.exists() or MMDET3D_DATA_LINK.is_symlink():
    print("MMDetection3D data link already exists:", MMDET3D_DATA_LINK)
    print("Resolved target:", MMDET3D_DATA_LINK.resolve())
else:
    MMDET3D_DATA_LINK.symlink_to(DATA_LINK, target_is_directory=True)
    print("Created MMDetection3D data symlink:")
    print(MMDET3D_DATA_LINK)
    print("->", MMDET3D_DATA_LINK.resolve())

MMDetection3D data link already exists: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes
Resolved target: /rs_scratch/users/ae04q066/nuscenes_full


In [11]:
print("MMDET3D_DATA_LINK exists:", MMDET3D_DATA_LINK.exists())
print("MMDET3D_DATA_LINK is symlink:", MMDET3D_DATA_LINK.is_symlink())
print("MMDET3D_DATA_LINK resolves to:", MMDET3D_DATA_LINK.resolve())
print()

for folder_name in EXPECTED_RAW_DIRS:
    folder_path: Path = MMDET3D_DATA_LINK / folder_name
    print(f"{folder_name:15s} exists={folder_path.exists()}  path={folder_path}")

MMDET3D_DATA_LINK exists: True
MMDET3D_DATA_LINK is symlink: True
MMDET3D_DATA_LINK resolves to: /rs_scratch/users/ae04q066/nuscenes_full

maps            exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/maps
samples         exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/samples
sweeps          exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/sweeps
v1.0-trainval   exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/v1.0-trainval


# Data preparation

https://mmdetection3d.readthedocs.io/en/latest/advanced_guides/datasets/nuscenes.html

The raw nuScenes dataset is available and linked under `data/nuscenes`, but the
processed metadata files required by MMDetection3D are not present yet.

In this step, I prepare the standard MMDetection3D nuScenes preprocessing command.

The command form follows the official MMDetection3D documentation:

`python tools/create_data.py nuscenes --root-path ./data/nuscenes --out-dir ./data/nuscenes --extra-tag nuscenes`

In my environment, I also set `PYTHONPATH` to the MMDetection3D repository root so that
Python can resolve imports from the `tools` package correctly.

In [12]:
CREATE_DATA_SCRIPT: Path = MMDET3D_ROOT / "tools" / "create_data.py"

print("MMDET3D_ROOT:", MMDET3D_ROOT)
print("CREATE_DATA_SCRIPT exists:", CREATE_DATA_SCRIPT.exists())
print("CREATE_DATA_SCRIPT:", CREATE_DATA_SCRIPT)

MMDET3D_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
CREATE_DATA_SCRIPT exists: True
CREATE_DATA_SCRIPT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/tools/create_data.py


In [13]:
CREATE_DATA_COMMAND: str = (
    "export PYTHONPATH=$(pwd):$PYTHONPATH && "
    "python tools/create_data.py "
    "nuscenes "
    "--root-path ./data/nuscenes "
    "--out-dir ./data/nuscenes "
    "--extra-tag nuscenes"
)

print("Run this command from:")
print(MMDET3D_ROOT)
print()
print(CREATE_DATA_COMMAND)

Run this command from:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d

export PYTHONPATH=$(pwd):$PYTHONPATH && python tools/create_data.py nuscenes --root-path ./data/nuscenes --out-dir ./data/nuscenes --extra-tag nuscenes


In [14]:
# Uncomment to run from the notebook

# !cd {MMDET3D_ROOT} && \
# export PYTHONPATH=$(pwd):$PYTHONPATH && \
# python tools/create_data.py nuscenes --root-path ./data/nuscenes --out-dir ./data/nuscenes --extra-tag nuscenes

In [15]:
# cd /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
# conda activate py38_mmdet3d
# export PYTHONPATH=$(pwd):$PYTHONPATH
# python tools/create_data.py nuscenes --root-path ./data/nuscenes --out-dir ./data/nuscenes --extra-tag nuscenes

# Verification of generated files

After running the preprocessing command, verify that the expected processed files
and database directory have been created.

In [16]:
print("Checking processed files in:", MMDET3D_DATA_LINK)
print()

for item_name in EXPECTED_PROCESSED_ITEMS:
    item_path: Path = MMDET3D_DATA_LINK / item_name
    print(f"{item_name:25s} exists={item_path.exists()}  path={item_path}")

Checking processed files in: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes

nuscenes_infos_train.pkl  exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_train.pkl
nuscenes_infos_val.pkl    exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_val.pkl
nuscenes_dbinfos_train.pkl exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_dbinfos_train.pkl
nuscenes_gt_database      exists=True  path=/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_gt_database


# Chosen baseline config

# Config changes

# Smoke test

# Full training launch

# Monitoring

# Evaluation

# First interpretation
Connect results to your EDA:
    • distance
    • category
    • pedestrians
small objects